In [1]:
import os
os.environ["KERAS_BACKEND"] = "torch"

In [2]:
import cvxpy as cp
import numpy as np
import keras
import keras.ops as K
from keras.layers import Input, Flatten, Dense, Lambda
from keras.optimizers import Adam
from keras.metrics import BinaryAccuracy

# from keras.models import Sequential
from deel.lip.model import Sequential

from deel.lip.layers import (
    SpectralDense,
    SpectralConv2D,
    ScaledL2NormPooling2D,
    FrobeniusDense,
)
from deel.lip.activations import GroupSort, GroupSort2
from deel.lip.losses import HKR, KR, HingeMargin, MulticlassHKR, MulticlassKR

In [3]:
from lipschitz_decomon_tools import echantillonner_boule_l2_simple, square_backward_bounds
import sys
sys.path.append('..')
from data_processing import load_data

In [4]:
x_train, x_test, y_train, y_test, y_test_ord = load_data("MNIST08")

In [5]:
print(y_test_ord[:10])

[1. 1. 1. 1. 1. 1. 0. 1. 1. 0.]


In [6]:
x_sample = x_test[6:7].flatten()

In [7]:
label = (y_train[6]).argmax()
print(label)

0


In [8]:
model_path = "/home/aws_install/robustess_project/lip_models/demo3_FC_vanilla_MNIST08_channelfirst_False_disj_Neurons_single_output.keras"
model = keras.models.load_model(model_path)
model.compile(
   
    loss=HKR(
        alpha=10.0, min_margin=1.0
    ),  # HKR stands for the hinge regularized KR loss
    metrics=[
        # KR,  # shows the KR term of the loss
        HingeMargin(min_margin=1.0),  # shows the hinge term of the loss
    ],
    optimizer=Adam(learning_rate=0.001),)

/home/aws_install/miniconda3/envs/k3torchenv/lib/python3.10/site-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'adam', because it has 12 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [9]:
# def square_backward_bounds(l, u, y):
#     # l (4,)
#     # u (4,)
#     # y (4,)

#     u = u - y
#     l = l - y

#     W = u + l #(4,)
#     # print(cp.multiply(-u,l).shape)
#     b =cp.sum(cp.multiply(-u,l)) - W@y #scalar
#     return W, np.array(b)[None]#(4,) & (1,)

In [10]:
def function_to_optimize_all(x, label, W_list, b_list, y_list, model, L=1):
    # function we want to optimize, combination of lipschitz constraints in all yi
    outputs = []
    for i in range(len(y_list)):
        if label == 0:
            output = model(y_list[i].reshape((1,28,28))[None]).cpu().detach().numpy()[0,0] +\
                L*cp.sqrt(W_list[i]@x+b_list[i]) #scalar
            outputs.append(output)
            # concave
        else:
            output = model(y_list[i].reshape((1,28,28))[None]).cpu().detach().numpy()[0,0] -\
                L*cp.sqrt(W_list[i]@x+b_list[i]) #scalar
            outputs.append(output)   
            # convexe
    if label == 0:
        # min(min(concaves)) -> min(concave) -> NON CONVEXE
        function = cp.min(cp.hstack(outputs)) 
    else:
        # min(max(convexes)) -> min(convexe) -> CONVEXE
        function = cp.max(cp.hstack(outputs))
    return function

In [11]:
eps = 0.1
nb_pts = 100

In [12]:
x = cp.Variable(784)

In [13]:
y_list = []
for _ in range(nb_pts):
        y_list.append(echantillonner_boule_l2_simple(x_sample, eps))

In [14]:
l = x_sample-eps
u = x_sample+eps
W_list = []
b_list = []
for y_i in y_list:
    W, b = square_backward_bounds(l,u,y_i)
    W_list.append(W)
    b_list.append(b)

In [15]:
constraints = [eps**2 - cp.norm(x - x_sample, 2)**2 >=0]
obj = cp.Maximize(function_to_optimize_all(x, label, W_list, b_list, y_list, model, L=1))

In [16]:
prob = cp.Problem(obj, constraints)
# prob.solve(solver='CLARABEL', verbose=True)  # Returns the optimal value.
# prob.solve(solver='ECOS', verbose=True)  # Returns the optimal value.
prob.solve(solver='SCS', verbose=True)  # Returns the optimal value.


(CVXPY) Jul 24 07:41:57 PM: Your problem has 784 variables, 1 constraints, and 0 parameters.
(CVXPY) Jul 24 07:41:57 PM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Jul 24 07:41:57 PM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Jul 24 07:41:57 PM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Jul 24 07:41:57 PM: Your problem is compiled with the CPP canonicalization backend.
(CVXPY) Jul 24 07:41:57 PM: Compiling problem (target solver=SCS).
(CVXPY) Jul 24 07:41:57 PM: Reduction chain: FlipObjective -> Dcp2Cone -> CvxAttr2Constr -> ConeMatrixStuffing -> SCS
(CVXPY) Jul 24 07:41:57 PM: Applying reduction FlipObjective
(CVXPY) Jul 24 07:41:57 PM: Applying reduction Dcp2Cone
(CVXPY) Jul 24 07:41:57 PM: Applying reduction CvxAttr2Constr
(CVXPY) Jul 24 07:41:57 PM: Applying reduction ConeMatrixStuffing


                                     CVXPY                                     
                                     v1.6.6                                    
-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------


(CVXPY) Jul 24 07:41:57 PM: Applying reduction SCS
(CVXPY) Jul 24 07:41:57 PM: Finished problem compilation (took 2.970e-01 seconds).
(CVXPY) Jul 24 07:41:57 PM: Invoking solver SCS  to obtain a solution.


-------------------------------------------------------------------------------
                                Numerical solver                               
-------------------------------------------------------------------------------
------------------------------------------------------------------
	       SCS v3.2.7 - Splitting Conic Solver
	(c) Brendan O'Donoghue, Stanford University, 2012
------------------------------------------------------------------
problem:  variables n: 887, constraints m: 1189
cones: 	  l: linear vars: 101
	  q: soc vars: 1088, qsize: 102
settings: eps_abs: 1.0e-05, eps_rel: 1.0e-05, eps_infeas: 1.0e-07
	  alpha: 1.50, scale: 1.00e-01, adaptive_scale: 1
	  max_iters: 100000, normalize: 1, rho_x: 1.00e-06
	  acceleration_lookback: 10, acceleration_interval: 10
lin-sys:  sparse-direct-amd-qdldl
	  nnz(A): 157889, nnz(P): 0
------------------------------------------------------------------
 iter | pri res | dua res |   gap   |   obj   |  scale  | time (s

  1750| 3.06e-04  8.93e-05  5.42e-04 -1.23e+00  3.24e-01  1.17e+00 
  2000| 3.64e-04  1.18e-05  1.06e-04 -1.23e+00  3.24e-01  1.32e+00 


(CVXPY) Jul 24 07:41:59 PM: Problem status: optimal
(CVXPY) Jul 24 07:41:59 PM: Optimal value: 1.227e+00
(CVXPY) Jul 24 07:41:59 PM: Compilation took 2.970e-01 seconds
(CVXPY) Jul 24 07:41:59 PM: Solver (including time spent in interface) took 1.599e+00 seconds


  2250| 2.55e-04  4.08e-05  2.82e-04 -1.23e+00  3.24e-01  1.48e+00 
  2450| 1.07e-07  1.85e-07  3.70e-07 -1.23e+00  3.24e-01  1.60e+00 
------------------------------------------------------------------
status:  solved
timings: total: 1.60e+00s = setup: 5.71e-02s + solve: 1.54e+00s
	 lin-sys: 1.45e+00s, cones: 1.07e-02s, accel: 3.92e-03s
------------------------------------------------------------------
objective = -1.227161
------------------------------------------------------------------
-------------------------------------------------------------------------------
                                    Summary                                    
-------------------------------------------------------------------------------


1.2271609661609526

## Emprircal Tests

In [17]:
x_tests = [echantillonner_boule_l2_simple(x_sample, eps) for _ in range(100)]

In [18]:
def function_to_optimize_all_np(x, label, W_list, b_list, y_list, model, L=1):
    # function we want to optimize, combination of lipschitz constraints in all yi
    outputs = []
    for i in range(len(y_list)):
        if label == 0:
            output = model(y_list[i].reshape((1,28,28))[None]).cpu().detach().numpy()[0,0] +\
                L*np.sqrt(W_list[i]@x+b_list[i]) #scalar
            outputs.append(output)
            # concave
        else:
            output = model(y_list[i].reshape((1,28,28))[None]).cpu().detach().numpy()[0,0] -\
                L*np.sqrt(W_list[i]@x+b_list[i]) #scalar
            outputs.append(output)   
            # convexe
    if label == 0:
        # min(min(concaves)) -> min(concave) -> NON CONVEXE
        function = np.min(outputs)
    else:
        # min(max(convexes)) -> min(convexe) -> CONVEXE
        function = np.max(outputs)
    return function

In [19]:
liste = [function_to_optimize_all_np(x, label, W_list, b_list, y_list, model, L=1) for x in x_tests]

In [20]:
max = np.max(liste)

In [21]:
max

1.2245915637770861

# Comparison with scipy solver

In [22]:
# from lipschitz_decomon_tools import get_local_maximum, function_to_optimize_all

In [23]:
# x_adv, f_adv = get_local_maximum(x_sample, label, eps, y_list, model)

In [24]:
# f_adv

In [25]:
# list2 = [function_to_optimize_all_np(x, label, W_list, b_list, y_list, model, L=1) for x in x_tests]

In [26]:
# min2 = np.min(list2)
# print(min2)

# Multiclass

In [27]:
def create_difference_model(base_model, label, i):
    """
    Crée et retourne un nouveau modèle Keras qui calcule la différence
    entre le logit du 'label' et le logit de 'i'.
    """
    entree_base = base_model.inputs
    sortie_logits_base = base_model.outputs[0]
    # print(label)
    # Définition de la couche Lambda avec la correction et un nom unique
    difference = Lambda(
        # lambda x, current_i=i: x[:, label] - x[:, current_i],
        lambda z: z[:, label:label+1] - z[:, i:i+1], 
        output_shape=(1,),
        # Nom de couche unique : très important !
        name=f"difference_{label}_vs_{i}"
    )(sortie_logits_base)

    # Création du modèle avec un nom unique
    difference_model = keras.Model(
        inputs=entree_base,
        outputs=difference,
        name=f"model_diff_{label}_vs_{i}"
    )
    
    return difference_model

In [28]:
def get_local_maximum(x_sample, label, eps, y_list, model, L=1):
    l = x_sample-eps
    u = x_sample+eps

    W_list = []
    b_list = []
    for y_i in y_list:
        W, b = square_backward_bounds(l,u,y_i)
        W_list.append(W)
        b_list.append(b)

    x = cp.Variable(784)
    
    constraints = [eps**2 - cp.norm(x - x_sample, 2)**2 >=0]

    # Run the optimizer
    if label == 0:
        obj = cp.Maximize(function_to_optimize_all(x, label, W_list, b_list, y_list, model, L=1))
    else:
        obj = cp.Minimize(function_to_optimize_all(x, label, W_list, b_list, y_list, model, L=1))

    prob = cp.Problem(obj, constraints)
    # prob.solve(solver='CLARABEL', verbose=True)  # Returns the optimal value.
    # prob.solve(solver='ECOS', verbose=True)  # Returns the optimal value.
    prob.solve(solver='SCS', verbose=True)  # Returns the optimal value.
    return prob.status, prob.value, x.value
    

In [36]:
def get_local_maximum_multiclass(x_sample, label, eps, y_list, model, L=1):
    """
    Adaptation du getlocalmaximum au cas multiclasse. On vient borner fgt - fi qui est une fonction racine de 2 lip
    """
    n_classes = model.output_shape[-1]
    print(type(range(n_classes)))
    list_outputs = list(range(n_classes))
    # print(list_outputs)
    list_outputs.remove(label)
    # print(list_outputs)
    # print(K.argsort(model(x.reshape((1,28,28))[None]))[:,-2])
    difference_model = create_difference_model(model, label, K.argsort(model(x_sample.reshape((1,28,28))[None]))[:,-2])

    return  get_local_maximum(x_sample, 1, eps, y_list, difference_model, L=np.sqrt(2)*L)   

In [37]:
x_train, x_test, y_train, y_test, y_test_ord = load_data("MNIST")

In [38]:
vanilla_model = keras.models.load_model("/home/aws_install/robustess_project/lip_models/demo0_vanilla_MNIST_channelfirst_False_disj_Neurons.keras")
vanilla_model.compile(
        # decreasing alpha and increasing min_margin improve robustness (at the cost of accuracy)
        # note also in the case of lipschitz networks, more robustness require more parameters.
        loss=MulticlassHKR(alpha=50, min_margin=0.05),
        optimizer=keras.optimizers.Adam(1e-3),
        metrics=["accuracy", MulticlassKR()],)

In [60]:
eps=0.1
nb_points = 100

In [61]:
x_sample = x_test[0:1].flatten()
label = y_test_ord[0]

In [62]:
y_list = [x_sample]
for i in range(nb_points):
    y_list.append(echantillonner_boule_l2_simple(x_sample, eps))

In [63]:
status, optimum, arg_optimum = get_local_maximum_multiclass(x_sample, label, eps, y_list, vanilla_model)

<class 'range'>


/home/aws_install/miniconda3/envs/k3torchenv/lib/python3.10/site-packages/keras/src/models/functional.py:238: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['input_layer']
Received: inputs=Tensor(shape=(1, 1, 28, 28))
  warnings.warn(msg)


(CVXPY) Jul 24 07:50:53 PM: Your problem has 784 variables, 1 constraints, and 0 parameters.
(CVXPY) Jul 24 07:50:53 PM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Jul 24 07:50:53 PM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Jul 24 07:50:53 PM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Jul 24 07:50:53 PM: Your problem is compiled with the CPP canonicalization backend.
(CVXPY) Jul 24 07:50:53 PM: Compiling problem (target solver=SCS).
(CVXPY) Jul 24 07:50:53 PM: Reduction chain: Dcp2Cone -> CvxAttr2Constr -> ConeMatrixStuffing -> SCS
(CVXPY) Jul 24 07:50:53 PM: Applying reduction Dcp2Cone
(CVXPY) Jul 24 07:50:53 PM: Applying reduction CvxAttr2Constr
(CVXPY) Jul 24 07:50:53 PM: Applying reduction ConeMatrixStuffing
(CVXPY) Jul 24 07:50:53 PM: Applying reduction SCS


                                     CVXPY                                     
                                     v1.6.6                                    
-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------


(CVXPY) Jul 24 07:50:54 PM: Finished problem compilation (took 2.786e-01 seconds).
(CVXPY) Jul 24 07:50:54 PM: Invoking solver SCS  to obtain a solution.


-------------------------------------------------------------------------------
                                Numerical solver                               
-------------------------------------------------------------------------------
------------------------------------------------------------------
	       SCS v3.2.7 - Splitting Conic Solver
	(c) Brendan O'Donoghue, Stanford University, 2012
------------------------------------------------------------------
problem:  variables n: 888, constraints m: 1193
cones: 	  l: linear vars: 102
	  q: soc vars: 1091, qsize: 103
settings: eps_abs: 1.0e-05, eps_rel: 1.0e-05, eps_infeas: 1.0e-07
	  alpha: 1.50, scale: 1.00e-01, adaptive_scale: 1
	  max_iters: 100000, normalize: 1, rho_x: 1.00e-06
	  acceleration_lookback: 10, acceleration_interval: 10
lin-sys:  sparse-direct-amd-qdldl
	  nnz(A): 158008, nnz(P): 0
------------------------------------------------------------------
 iter | pri res | dua res |   gap   |   obj   |  scale  | time (s

(CVXPY) Jul 24 07:50:58 PM: Problem status: optimal
(CVXPY) Jul 24 07:50:58 PM: Optimal value: -3.023e-01
(CVXPY) Jul 24 07:50:58 PM: Compilation took 2.786e-01 seconds
(CVXPY) Jul 24 07:50:58 PM: Solver (including time spent in interface) took 4.622e+00 seconds


  7250| 1.65e-05  6.12e-07  1.56e-05 -3.02e-01  8.75e-01  4.50e+00 
  7450| 5.21e-07  1.52e-06  9.06e-07 -3.02e-01  8.75e-01  4.62e+00 
------------------------------------------------------------------
status:  solved
timings: total: 4.62e+00s = setup: 5.55e-02s + solve: 4.56e+00s
	 lin-sys: 4.26e+00s, cones: 3.45e-02s, accel: 1.10e-02s
------------------------------------------------------------------
objective = -0.302282
------------------------------------------------------------------
-------------------------------------------------------------------------------
                                    Summary                                    
-------------------------------------------------------------------------------


In [64]:
print(optimum, label)

-0.3022813970025564 7
